# Análise de Homicídios Intencionais Globais

## Introdução

Este notebook apresenta uma análise abrangente dos dados de homicídios intencionais globais, utilizando dados do Escritório das Nações Unidas sobre Drogas e Crime (UNODC). O estudo abrange limpeza de dados, análise exploratória, respostas a perguntas estatísticas e modelagem de regressão para previsão de tendências futuras.

**Fonte dos dados:** UNODC - United Nations Office on Drugs and Crime

**Dataset:** `data/data_cts_intentional_homicide.xlsx`

## Objetivo

O objetivo deste trabalho é realizar uma análise completa dos dados de homicídios intencionais ao redor do mundo, respondendo a perguntas estatísticas relevantes e construindo modelos preditivos para estimar tendências futuras. Este notebook serve como entregável acadêmico para a disciplina de Tópicos Especiais em Computação I.

### Etapas da análise:
1. Limpeza e preparação dos dados
2. Análise Exploratória de Dados (EDA)
3. Respostas às perguntas estatísticas
4. Modelagem de regressão e previsões
5. Conclusões

## Configuração do Ambiente

Configuração do caminho do projeto e importação dos módulos necessários.

In [ ]:
import sys
import os

# Adicionar o diretório raiz do projeto ao path
sys.path.insert(0, os.path.abspath('..'))

# Importações dos módulos do projeto
from src.cleaning import run_cleaning_pipeline
from src.eda import generate_all_charts, answer_all_questions
from src.regression import run_regression_pipeline
from src.utils import load_cleaned_data

import pandas as pd
import plotly.io as pio

# Configurar Plotly para exibição inline
pio.renderers.default = 'notebook'

print('Módulos importados com sucesso!')

## Limpeza de Dados

Nesta etapa, realizamos a limpeza completa do dataset bruto. O pipeline de limpeza executa as seguintes operações:

1. **Carregamento** do arquivo Excel original
2. **Padronização** dos nomes das colunas para snake_case
3. **Filtragem** por indicador (Vítimas de homicídio intencional)
4. **Filtragem** por unidade de medida (Taxa por 100.000 habitantes)
5. **Remoção** de agregações globais (World, Global, Various, Unknown)
6. **Remoção** de valores nulos em colunas essenciais
7. **Conversão** de tipos de dados (ano para inteiro, valor para float)
8. **Exportação** dos datasets limpos para CSV

In [ ]:
# Executar o pipeline completo de limpeza
df = run_cleaning_pipeline('data/data_cts_intentional_homicide.xlsx', 'outputs')

print(f'Dataset limpo: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'\nColunas: {list(df.columns)}')
print(f'\nPeríodo: {df["year"].min()} - {df["year"].max()}')
print(f'Países únicos: {df["country"].nunique()}')

df.head(10)

## Análise Exploratória de Dados (EDA)

A análise exploratória permite identificar padrões, distribuições e tendências nos dados de homicídios globais. Geramos os seguintes gráficos interativos:

- **Histograma**: Distribuição das taxas de homicídio
- **Boxplot**: Distribuição por região geográfica
- **Gráfico de linhas**: Tendências temporais dos países com maiores taxas
- **Gráfico de barras**: Ranking dos 10 países com maiores taxas médias
- **Heatmap**: Correlação entre variáveis numéricas

In [ ]:
# Gerar todos os gráficos e salvar como PNG
generate_all_charts(df, 'images')
print('Gráficos salvos na pasta images/')

### Distribuição das Taxas de Homicídio

O histograma abaixo mostra a distribuição das taxas de homicídio por 100.000 habitantes em todos os países e anos disponíveis.

In [ ]:
from src.eda import plot_histogram

fig = plot_histogram(df, 'value', 'Distribuição das Taxas de Homicídio')
fig.show()

### Taxas de Homicídio por Região

O boxplot permite comparar a distribuição das taxas entre diferentes regiões geográficas, identificando outliers e medianas.

In [ ]:
from src.eda import plot_boxplot, _filter_aggregate

df_agg = _filter_aggregate(df, 'Total')
fig = plot_boxplot(df_agg, 'region', 'value', 'Taxas de Homicídio por Região')
fig.show()

### Tendências Temporais — Top 5 Países

O gráfico de linhas mostra a evolução temporal das taxas de homicídio nos 5 países com as maiores taxas médias.

In [ ]:
from src.eda import plot_line_chart

# Selecionar top 5 países por taxa média
top_countries = (
    df_agg.groupby('country')['value']
    .mean()
    .nlargest(5)
    .index.tolist()
)
df_top = df_agg[df_agg['country'].isin(top_countries)]

fig = plot_line_chart(df_top, 'year', 'value', 'country', 'Tendência de Homicídios — Top 5 Países')
fig.show()

### Top 10 Países por Taxa Média de Homicídio

O gráfico de barras apresenta os 10 países com as maiores taxas médias de homicídio ao longo de todo o período disponível.

In [ ]:
from src.eda import plot_bar_chart

top10 = (
    df_agg.groupby('country')['value']
    .mean()
    .nlargest(10)
    .reset_index()
)
top10.columns = ['country', 'avg_rate']

fig = plot_bar_chart(top10, 'country', 'avg_rate', 'Top 10 Países por Taxa Média de Homicídio')
fig.show()

### Matriz de Correlação

O heatmap mostra as correlações entre as variáveis numéricas do dataset.

In [ ]:
from src.eda import plot_heatmap

fig = plot_heatmap(df, 'Matriz de Correlação — Variáveis Numéricas')
fig.show()

## Perguntas Estatísticas

Nesta seção, respondemos às 10 perguntas estatísticas obrigatórias utilizando funções implementadas no módulo `src/eda.py`.

In [ ]:
# Executar todas as perguntas de uma vez
respostas = answer_all_questions(df)
print('Todas as perguntas foram respondidas com sucesso!')

### Pergunta 1
**Quais são os 10 países com maior taxa média de homicídio nos últimos 5 anos disponíveis?**

In [ ]:
print('Top 10 países por taxa média de homicídio (últimos 5 anos):')
respostas['q1']

### Pergunta 2
**Quais são os 10 países com maior taxa de homicídio feminino em 2022?**

In [ ]:
print('Top 10 países por taxa de homicídio feminino em 2022:')
respostas['q2']

### Pergunta 3
**Quais regiões possuem os maiores totais de homicídios?**

In [ ]:
print('Regiões com maiores totais de homicídios:')
respostas['q3']

### Pergunta 4
**Quais países possuem a menor taxa de homicídio por sub-região?**

In [ ]:
print('Países com menor taxa de homicídio por sub-região:')
respostas['q4']

### Pergunta 5
**Quais países possuem as menores taxas de homicídio feminino?**

In [ ]:
print('Países com menores taxas de homicídio feminino:')
respostas['q5']

### Pergunta 6
**Quais sub-regiões possuem os maiores totais de homicídios?**

In [ ]:
print('Sub-regiões com maiores totais de homicídios:')
respostas['q6']

### Pergunta 7
**Qual país possui a maior taxa de homicídio por continente em 2020?**

In [ ]:
print('País com maior taxa de homicídio por continente em 2020:')
respostas['q7']

### Pergunta 8
**Qual é o país mais violento para mulheres em 2021?**

In [ ]:
print('País mais violento para mulheres em 2021:')
respostas['q8']

### Pergunta 9
**Qual país possui o maior valor individual de indicador em todos os anos?**

In [ ]:
print('País com maior valor individual de indicador:')
respostas['q9']

### Pergunta 10
**Qual é a taxa média de homicídio do Brasil nos últimos 10 anos disponíveis?**

In [ ]:
print(f'Taxa média de homicídio do Brasil (últimos 10 anos): {respostas["q10"]:.2f} por 100.000 habitantes')

## Análise de Regressão

Nesta seção, construímos modelos de regressão para prever as taxas futuras de homicídio no Brasil. Utilizamos dois modelos:

1. **Regressão Linear**: Modelo simples que assume relação linear entre ano e taxa
2. **Random Forest**: Modelo ensemble que captura relações não-lineares

Os modelos são treinados com dados históricos (80% treino, 20% teste) e geram previsões para os anos de 2023 a 2026.

In [ ]:
# Carregar dataset de regressão
df_reg = load_cleaned_data('outputs/dataset_regressao.csv')
print(f'Dataset de regressão: {df_reg.shape[0]} registros')
print(f'Países disponíveis: {df_reg["country"].nunique()}')
df_reg.head()

### Pipeline de Regressão para o Brasil

Executamos o pipeline completo de regressão para o Brasil, que inclui:
- Preparação dos dados (filtragem por país, split treino/teste)
- Treinamento dos modelos (Linear Regression e Random Forest)
- Avaliação de métricas (MAE, MSE, RMSE, R²)
- Geração de previsões para 2023-2026
- Visualização dos resultados

In [ ]:
# Executar pipeline de regressão para o Brasil
resultados = run_regression_pipeline(df_reg, 'Brazil', 'images')

print('=== Resultados da Regressão para o Brasil ===')
print(f'\nRegressão Linear:')
for metric, value in resultados['linear_regression'].items():
    print(f'  {metric}: {value:.4f}')

print(f'\nRandom Forest:')
for metric, value in resultados['random_forest'].items():
    print(f'  {metric}: {value:.4f}')

### Previsões para 2023-2026

In [ ]:
print('Previsões - Regressão Linear:')
display(resultados['predictions_lr'])

print('\nPrevisões - Random Forest:')
display(resultados['predictions_rf'])

### Visualização: Dados Históricos vs Previsões

In [ ]:
from src.regression import plot_historical_and_predictions, plot_real_vs_predicted
from src.regression import prepare_regression_data

# Dados históricos do Brasil
df_brazil = df_reg[df_reg['country'] == 'Brazil'][['year', 'value']]

# Gráfico: Histórico + Previsões (Regressão Linear)
fig = plot_historical_and_predictions(
    df_brazil, resultados['predictions_lr'],
    'Brasil - Regressão Linear: Histórico vs Previsões'
)
fig.show()

In [ ]:
# Gráfico: Histórico + Previsões (Random Forest)
fig = plot_historical_and_predictions(
    df_brazil, resultados['predictions_rf'],
    'Brasil - Random Forest: Histórico vs Previsões'
)
fig.show()

### Visualização: Valores Reais vs Preditos

In [ ]:
from src.regression import train_linear_regression, train_random_forest, evaluate_model

# Preparar dados para visualização real vs predito
X_train, X_test, y_train, y_test = prepare_regression_data(df_reg, 'Brazil')

lr_model = train_linear_regression(X_train, y_train)
rf_model = train_random_forest(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

# Gráfico: Real vs Predito (Regressão Linear)
fig = plot_real_vs_predicted(y_test, y_pred_lr, 'Brasil - Regressão Linear: Real vs Predito')
fig.show()

In [ ]:
# Gráfico: Real vs Predito (Random Forest)
fig = plot_real_vs_predicted(y_test, y_pred_rf, 'Brasil - Random Forest: Real vs Predito')
fig.show()

## Conclusões

A partir da análise realizada neste notebook, podemos destacar as seguintes conclusões:

### Principais Achados

1. **Distribuição desigual**: As taxas de homicídio apresentam grande variação entre países e regiões, com concentração significativa em determinadas áreas geográficas.

2. **Padrões regionais**: Algumas regiões, particularmente nas Américas e na África, apresentam taxas consistentemente mais elevadas que outras regiões do mundo.

3. **Tendências temporais**: A análise temporal revela que alguns países apresentam tendências de redução nas taxas de homicídio, enquanto outros mantêm níveis elevados ao longo dos anos.

4. **Violência de gênero**: A análise específica de homicídios femininos revela padrões distintos que merecem atenção especial em políticas públicas.

5. **Modelos preditivos**: Os modelos de regressão treinados para o Brasil permitem estimar tendências futuras, sendo o Random Forest geralmente mais adequado para capturar variações não-lineares nos dados.

### Limitações

- Os dados dependem da qualidade dos registros oficiais de cada país
- A previsão utiliza apenas o ano como variável preditora, não considerando fatores socioeconômicos
- Alguns países podem ter dados incompletos em determinados períodos

### Trabalhos Futuros

- Incorporar variáveis socioeconômicas nos modelos de regressão
- Aplicar modelos de séries temporais (ARIMA, Prophet)
- Realizar análise de clusters para agrupar países com padrões similares
- Expandir a análise para incluir outros tipos de crimes violentos

---

*Trabalho desenvolvido para a disciplina de Tópicos Especiais em Computação I.*